In [ ]:
"""
PPE Vision System — YOLOv8 Safety Detection
============================================
Production-grade training and inference pipeline for Personal Protective
Equipment (PPE) detection on construction/factory sites.

Architecture:
    - Backbone : YOLOv8n (anchor-free, CSPDarknet)
    - Dataset  : Roboflow PPE Detection dataset
    - Deployment: FastAPI + Docker (CPU inference)

Author  : Alejandro Toro Arrabal
Version : 2.0.0
"""

# ── Standard Library
import os
import sys
import logging
import random
import subprocess
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional
from base64 import b64encode
from contextlib import contextmanager

# ── Third Party
import cv2
import numpy as np
import torch

# ── Colab-specific imports (graceful fallback for non-Colab environments)
try:
    from google.colab import userdata
    from IPython.display import HTML, display
    IN_COLAB = True

except ImportError:
    IN_COLAB = False

# ── Dependency check — install only if missing
def _ensure_dependencies() -> None:
    """Verifies and installs required packages if not present."""

    required = {"ultralytics": "ultralytics", "roboflow": "roboflow"}

    for module, package in required.items():
        try:
            __import__(module)

        except ImportError:
            print(f"[SYSTEM] Installing {package}...")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", package, "-q"]
            )

_ensure_dependencies()

from ultralytics import YOLO 
from roboflow import Roboflow  

In [ ]:
# ════════════════════════════════════════════════════════
# 1. LOGGING INFRASTRUCTURE
# ════════════════════════════════════════════════════════

def setup_logger(name: str = "PPE_YOLO") -> logging.Logger:
    """
    Configures a structured logger.
    Clears existing handlers to prevent duplicate output on notebook reruns.
    """
    logger = logging.getLogger(name)

    if logger.hasHandlers():
        logger.handlers.clear()

    logger.setLevel(logging.INFO)
    logger.propagate = False # Prevents Colab's hidden logger from duplicating output

    formatter = logging.Formatter(
        fmt="%(asctime)s - [%(levelname)s] - %(message)s",
        datefmt="%H:%M:%S"
    )

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger


logger = setup_logger()

In [ ]:
# ════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ════════════════════════════════════════════════════════

@dataclass(frozen=True)
class YOLOConfig:
    """
    Immutable configuration container for the PPE Vision System.

    Frozen dataclass enforces that no hyperparameter is accidentally
    mutated during the training or inference lifecycle.

    API_KEY is intentionally excluded from the dataclass —
    secrets are never stored in config objects. Pass at runtime via
    environment variable: ROBOFLOW_API_KEY.
    """

    # ── Identity & Data
    PROJECT_NAME: str = "ppe_production_v2"
    ROBOFLOW_WORKSPACE: str = "testcasque"
    ROBOFLOW_PROJECT: str = "ppe-detection-qlq3d"
    ROBOFLOW_VERSION: int = 1

    # ── Paths
    BASE_OUTPUT_DIR: Path = field(
        default_factory=lambda: Path.cwd().resolve() / "runs" / "detect"
    )

    INPUT_VIDEO: Path = field(
        default_factory=lambda: Path("construction_site.mp4")
    )

    OUTPUT_VIDEO: Path = field(
        default_factory=lambda: Path("ppe_result.mp4")
    )

    # ── Hyperparameters
    EPOCHS: int = 100
    BATCH_SIZE: int = 16
    IMG_SIZE: int = 640
    CONF_THRESHOLD: float = 0.40
    IOU_THRESHOLD: float = 0.50

    # ── Reproducibility
    SEED: int = 42

In [ ]:
# ════════════════════════════════════════════════════════
# 3. PPE VISION SYSTEM — OOP CONTROLLER
# ════════════════════════════════════════════════════════
class PPEVisionSystem:
    """
    End-to-end controller for the PPE Safety Detection pipeline.

    Responsibilities:
        - Dataset ingestion via Roboflow API
        - YOLOv8n training with deterministic seed
        - Artifact management (auto-locate best.pt)
        - API-ready inference (accepts numpy arrays, returns JSON-serializable dict)
        - Offline video stream processing
        - Colab visualization utility

    Usage:
        config = YOLOConfig(EPOCHS=100)
        system = PPEVisionSystem(config)
        system.train()
        system.load_model()
        result = system.predict_image(image_array)
    """

    def __init__(self, config: YOLOConfig) -> None:
        self.cfg = config
        self.model: Optional[YOLO] = None
        self.device: str = "cuda:0" if torch.cuda.is_available() else "cpu"

        self._set_seed()
        logger.info(f"PPEVisionSystem initialized | Device: {self.device.upper()}")


    # ────────────────────────────────────────────────────
    # PRIVATE UTILITIES
    # ────────────────────────────────────────────────────
    def _set_seed(self) -> None:
        """
        Locks all random number generators for full reproducibility.

        Required components:
            - os.environ PYTHONHASHSEED : Python dict hash randomization
            - random                    : Python stdlib random
            - numpy                     : NumPy operations
            - torch.manual_seed         : CPU tensor ops
            - torch.cuda.manual_seed_all: All GPU tensor ops
            - cudnn.deterministic       : Deterministic CUDA kernels
            - cudnn.benchmark = False   : Disable auto-tuner (non-deterministic)

        Note: deterministic=True incurs ~10-20% training speed penalty.
        Disable in production inference by setting benchmark=True.
        """
        os.environ["PYTHONHASHSEED"] = str(self.cfg.SEED)
        random.seed(self.cfg.SEED)
        np.random.seed(self.cfg.SEED)
        torch.manual_seed(self.cfg.SEED)
        torch.cuda.manual_seed_all(self.cfg.SEED)

        if torch.cuda.is_available():
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

        logger.info(f"Deterministic seed locked: {self.cfg.SEED}")

    def _get_best_artifact(self) -> Optional[Path]:
        """
        Locates the most recently modified best.pt artifact.

        Scans BASE_OUTPUT_DIR recursively and returns the latest file
        by modification timestamp. Returns None if no artifact exists.
        """
        if not self.cfg.BASE_OUTPUT_DIR.exists():
            return None

        candidates: List[Path] = list(
            self.cfg.BASE_OUTPUT_DIR.rglob("best.pt")
        )
        if not candidates:
            return None

        return sorted(candidates, key=lambda p: p.stat().st_mtime)[-1]


    # ────────────────────────────────────────────────────
    # PUBLIC PIPELINE METHODS
    # ────────────────────────────────────────────────────
    def train(self) -> None:
        """
        Executes the Roboflow dataset ingestion and YOLOv8 training pipeline.

        Pipeline:
            1. Authenticate with Roboflow API (key from environment variable)
            2. Download dataset in YOLOv8 format
            3. Initialize YOLOv8n backbone with pretrained ImageNet weights
            4. Execute training with configured hyperparameters

        Raises:
            EnvironmentError : If ROBOFLOW_API_KEY is not set
            RuntimeError     : If training fails for any reason
        """
        api_key = os.environ.get("ROBOFLOW_API_KEY")

        if not api_key:
            raise EnvironmentError(
                "ROBOFLOW_API_KEY environment variable not set. "
                "Export it before running: export ROBOFLOW_API_KEY=your_key"
            )

        logger.info("Initiating MLOps Training Protocol...")
        logger.info(
            f"Config | Epochs: {self.cfg.EPOCHS} | "
            f"Batch: {self.cfg.BATCH_SIZE} | "
            f"ImgSize: {self.cfg.IMG_SIZE}"
        )

        try:
            # ── Dataset ingestion
            logger.info("Connecting to Roboflow API...")
            rf = Roboflow(api_key=api_key)

            project = rf.workspace(self.cfg.ROBOFLOW_WORKSPACE).project(
                self.cfg.ROBOFLOW_PROJECT
            )

            dataset = project.version(self.cfg.ROBOFLOW_VERSION).download("yolov8")
            logger.info(f"Dataset downloaded: {dataset.location}")

            # ── Model initialization
            trainer = YOLO("yolov8n.pt")
            logger.info("YOLOv8n backbone loaded with pretrained weights.")

            # ── Training execution
            trainer.train(
                data=str(Path(dataset.location) / "data.yaml"),
                epochs=self.cfg.EPOCHS,
                imgsz=self.cfg.IMG_SIZE,
                batch=self.cfg.BATCH_SIZE,
                project=str(self.cfg.BASE_OUTPUT_DIR),
                name=self.cfg.PROJECT_NAME,
                exist_ok=True,
                seed=self.cfg.SEED,
                verbose=False,
            )

            artifact = self._get_best_artifact()
            logger.info(f"Training complete. Best artifact: {artifact}")

        except Exception as e:

            logger.error(f"Training pipeline failed: {e}")
            raise RuntimeError(f"Training aborted: {e}") from e

    def load_model(self) -> None:
        """
        Loads the best model artifact into memory and runs warmup inference.

        Warmup inference compiles CUDA kernels on first pass — without it,
        the first real inference call is artificially slow (~3-5x latency spike).
        Using a dummy zero array, no real image required for warmup.

        Raises:
            FileNotFoundError: If no best.pt artifact exists.
        """
        artifact_path = self._get_best_artifact()

        if artifact_path is None:
            raise FileNotFoundError(
                "No model artifact found. Execute train() before load_model()."
            )

        self.model = YOLO(artifact_path)
        logger.info(
            f"Artifact loaded: {artifact_path.parent.name}/{artifact_path.name}"
        )

        # ── Warmup pass — compiles CUDA kernels, stabilizes first inference latency
        logger.info("Running warmup inference...")

        dummy_frame = np.zeros(
            (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE, 3), dtype=np.uint8
        )

        self.model.predict(source=dummy_frame, verbose=False)
        logger.info(f"Model ready on {self.device.upper()}.")

    def predict_image(self, image: np.ndarray) -> Dict[str, Any]:
        """
        API-ready inference method. Accepts numpy arrays, compatible with
        FastAPI's in-memory byte decoding pipeline (cv2.imdecode).

        Args:
            image: RGB numpy array of shape (H, W, 3), dtype uint8.
                   Note: OpenCV reads BGR, convert before passing:
                   image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        Returns:
            JSON-serializable dict:
            {
                "total_detections": int,
                "detections": [
                    {
                        "class_id": int,
                        "class_name": str,
                        "confidence": float,
                        "bbox": [xmin, ymin, xmax, ymax]
                    },
                    ...
                ]
            }

        Raises:
            ValueError      : If image is None or empty.
            RuntimeError    : If model is not loaded.
        """
        # ── Input validation
        if image is None or image.size == 0:

            raise ValueError(
                "Invalid image input, array is None or empty. "
                "Ensure bytes were decoded correctly via cv2.imdecode."
            )

        # ── Lazy model loading
        if self.model is None:
            self.load_model()

        # ── Inference
        results = self.model.predict(
            source=image,
            conf=self.cfg.CONF_THRESHOLD,
            iou=self.cfg.IOU_THRESHOLD,
            verbose=False,
        )[0]

        # ── Parse detections into JSON-serializable payload
        detections: List[Dict[str, Any]] = []
        for box in results.boxes:
            detections.append({
                "class_id"  : int(box.cls[0]),
                "class_name": self.model.names[int(box.cls[0])],
                "confidence": round(float(box.conf[0]), 4),
                "bbox"      : [round(float(x), 2) for x in box.xyxy[0]],
                # [xmin, ymin, xmax, ymax] — pixel coordinates
            })

        return {
            "total_detections": len(detections),
            "detections"      : detections,
        }

    def process_video_stream(self) -> None:
        """
        Offline QA tool for annotating MP4 video files with PPE detections.

        Reads INPUT_VIDEO frame by frame, runs YOLOv8 inference on each frame,
        draws bounding boxes, and writes the annotated output to OUTPUT_VIDEO.

        Logs progress every 50 frames to avoid log flooding.
        """
        if self.model is None:
            self.load_model()

        if not self.cfg.INPUT_VIDEO.exists():
            logger.error(f"Video source not found: {self.cfg.INPUT_VIDEO}")
            return

        cap = cv2.VideoCapture(str(self.cfg.INPUT_VIDEO))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # H.264 codec, wider compatibility than mp4v
        writer = cv2.VideoWriter(
            str(self.cfg.OUTPUT_VIDEO),
            cv2.VideoWriter_fourcc(*"avc1"),
            fps,
            (width, height),
        )

        logger.info(
            f"Processing video | Frames: {total_frames} | "
            f"FPS: {fps} | Resolution: {width}x{height}"
        )

        frame_idx = 0
        try:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                results = self.model.predict(
                    source=frame,
                    conf=self.cfg.CONF_THRESHOLD,
                    iou=self.cfg.IOU_THRESHOLD,
                    verbose=False,
                )
                writer.write(results[0].plot())

                frame_idx += 1
                if frame_idx % 50 == 0:
                    progress = (frame_idx / total_frames) * 100
                    logger.info(f"Progress: {progress:.1f}% ({frame_idx}/{total_frames})")

        finally:

            # Always release resources — even if an exception occurs mid-video
            cap.release()
            writer.release()

        logger.info(f"Video processing complete: {self.cfg.OUTPUT_VIDEO}")

    def display_video(self) -> Optional[Any]:
        """
        Colab/Jupyter utility — renders the output video inline.

        Returns None silently if not in a notebook environment or if
        the output video has not been generated yet.
        """
        if not IN_COLAB:
            logger.warning("display_video() is only available in Colab/Jupyter.")
            return None

        if not self.cfg.OUTPUT_VIDEO.exists():
            logger.warning(
                f"Output video not found: {self.cfg.OUTPUT_VIDEO}. "
                "Run process_video_stream() first."
            )
            return None

        with open(self.cfg.OUTPUT_VIDEO, "rb") as f:
            mp4_bytes = f.read()

        data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
        return HTML(
            f'<video width=640 controls autoplay loop>'
            f'<source src="{data_url}" type="video/mp4">'
            f"</video>"
        )




In [ ]:
# ════════════════════════════════════════════════════════
# 4. EXECUTION ENTRYPOINT
# ════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Secure API key retrieval
    # In Colab: store key in Secrets under "Roboflow_key"
    # In local/CI: export ROBOFLOW_API_KEY=your_key
    if IN_COLAB:
        os.environ["ROBOFLOW_API_KEY"] = userdata.get("Roboflow_key")

    api_key_present = bool(os.environ.get("ROBOFLOW_API_KEY"))
    if not api_key_present:
        logger.error(
            "ROBOFLOW_API_KEY not found. "
            "Set it via Colab Secrets or: export ROBOFLOW_API_KEY=your_key"
        )
        sys.exit(1)

    # ── System configuration
    config = YOLOConfig(
        EPOCHS=100,
        BATCH_SIZE=16,
        CONF_THRESHOLD=0.40,
    )

    system = PPEVisionSystem(config)

    # ── Phase 1: Training (skipped if artifact already exists)
    artifact = system._get_best_artifact()
    if artifact is None:
        logger.info("No artifact found — initiating training pipeline.")
        system.train()
    else:
        logger.info(f"Artifact found at {artifact} — skipping training.")

    # ── Phase 2: Model loading
    system.load_model()

    # ── Phase 3: API smoke test
    # Validates the full inference pipeline with a dummy frame
    logger.info("Running API smoke test...")
    dummy_image = np.zeros((640, 640, 3), dtype=np.uint8)
    smoke_result = system.predict_image(dummy_image)

    assert isinstance(smoke_result["total_detections"], int), \
        "Smoke test failed: total_detections is not an int"

    assert isinstance(smoke_result["detections"], list), \
        "Smoke test failed: detections is not a list"

    logger.info(f"Smoke test passed. Output schema: {list(smoke_result.keys())}")

    # ── Phase 4: Video processing (only if source video exists)
    if config.INPUT_VIDEO.exists():
        system.process_video_stream()
        if IN_COLAB:
            display(system.display_video())
    else:
        logger.info(
            f"Input video not found ({config.INPUT_VIDEO}). "
            "Skipping video processing phase."
        )